In [1]:
import logging

import pandas as pd
from pandas import DataFrame

import numpy as np
from numpy import random
import gensim
import nltk

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, \
confusion_matrix

from sklearn.feature_extraction.text import CountVectorizer, \
TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

import matplotlib.pyplot as plt
from nltk.corpus import stopwords
import re
from bs4 import BeautifulSoup
%matplotlib inline

In [2]:
df = pd.read_csv('final_movie_soundtrack_data_v2.csv')
del df['Unnamed: 0']

In [3]:
labels = ['action', 'comedy', 'drama', 'horror','romance', 'sci-fi']

In [4]:
df.columns

Index(['track_name', 'album_name', 'artist_names', 'release_date',
       'duration_ms', 'popularity', 'song_genres', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'title', 'movie_genre', 'action', 'comedy', 'drama', 'horror',
       'romance', 'sci-fi'],
      dtype='object')

In [5]:
df.head()

,track_name,album_name,artist_names,release_date,duration_ms,popularity,song_genres,danceability,energy,key,...,tempo,time_signature,title,movie_genre,action,comedy,drama,horror,romance,sci-fi
0,Space Exploration,Venom,Ludwig Göransson,10/5/18,263973,21,"nordic soundtrack,orchestral soundtrack,soundt...",0.194,0.294,8,...,159.468,4,Venom,"Action,Sci-Fi",1,0,0,0,0,1
1,Symbiotes Arrive,Venom,Ludwig Göransson,10/5/18,123040,17,"nordic soundtrack,orchestral soundtrack,soundt...",0.440,0.177,0,...,124.030,4,Venom,"Action,Sci-Fi",1,0,0,0,0,1
2,First Contact,Venom,Ludwig Göransson,10/5/18,209586,15,"nordic soundtrack,orchestral soundtrack,soundt...",0.194,0.153,0,...,117.213,4,Venom,"Action,Sci-Fi",1,0,0,0,0,1
3,Eddie's Blues,Venom,Ludwig Göransson,10/5/18,290440,20,"nordic soundtrack,orchestral soundtrack,soundt...",0.224,0.205,1,...,119.046,4,Venom,"Action,Sci-Fi",1,0,0,0,0,1
4,"Run, Eddie, Run",Venom,Ludwig Göransson,10/5/18,107546,20,"nordic soundtrack,orchestral soundtrack,soundt...",0.154,0.670,10,...,84.224,5,Venom,"Action,Sci-Fi",1,0,0,0,0,1


In [6]:
df.columns

Index(['track_name', 'album_name', 'artist_names', 'release_date',
       'duration_ms', 'popularity', 'song_genres', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'title', 'movie_genre', 'action', 'comedy', 'drama', 'horror',
       'romance', 'sci-fi'],
      dtype='object')

# Drop features we don't want to use
df = df.drop(columns=['track_name', 'album_name', 'artist_names', 'release_date',\
             'popularity', 'song_genres','key','title', 'time_signature', 'movie_genre'
             ]
            )

In [7]:
df = df.drop(columns=['track_name', 'album_name', 'artist_names', 'release_date',\
                      'song_genres','key','title', 'movie_genre'
             ]
            )

In [8]:
df.columns

Index(['duration_ms', 'popularity', 'danceability', 'energy', 'loudness',
       'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'action', 'comedy', 'drama',
       'horror', 'romance', 'sci-fi'],
      dtype='object')

In [9]:
df_action = df.drop(columns=['comedy', 'drama', 'horror','romance', 'sci-fi'])
df_comedy = df.drop(columns=['action', 'drama', 'horror','romance', 'sci-fi'])
df_drama = df.drop(columns=['action','comedy', 'horror','romance', 'sci-fi'])
df_horror = df.drop(columns=['action','comedy', 'drama', 'romance', 'sci-fi'])
df_romance = df.drop(columns=['action','comedy', 'drama', 'horror', 'sci-fi'])
df_sci_fi = df.drop(columns=['action','comedy', 'drama', 'horror','romance'])

In [10]:
print(df_action.action.value_counts())
print(df_drama.drama.value_counts())
print(df_horror.horror.value_counts())
print(df_drama.drama.value_counts())
print(df_romance.romance.value_counts())
print(df_sci_fi['sci-fi'].value_counts())

action
1    1224
0    1170
Name: count, dtype: int64
drama
0    1473
1     921
Name: count, dtype: int64
horror
0    2106
1     288
Name: count, dtype: int64
drama
0    1473
1     921
Name: count, dtype: int64
romance
0    2111
1     283
Name: count, dtype: int64
sci-fi
0    1581
1     813
Name: count, dtype: int64


In [11]:
def run_naive_bayes(df, y_label):

    
    X = df.drop(columns=y_label)
    trans = MinMaxScaler()
    X = trans.fit_transform(X)
    #X = DataFrame(X)
    
    y = df[y_label]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = 42)
    
    nb_classifier = MultinomialNB()
    nb_classifier.fit(X_train, y_train)
    
    predictions = nb_classifier.predict(X_test)
    
    accuracy = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, average='macro')
    print("Test Accuracy/F1 for {}: {:.2f}% / {}".format(y_label, accuracy * 100,f1))

    #print("\nClassification Report:\n", classification_report(y_test, predictions))
    #print("\nConfusion Matrix:\n", confusion_matrix(y_test, predictions))
    
    return [nb_classifier, predictions]

In [12]:
def run_rf(df, y_label):
    X = df.drop(columns=y_label)
    #trans = MinMaxScaler()
    trans = RobustScaler()
    X = trans.fit_transform(X)
    #X = DataFrame(X)
    
    y = df[y_label]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = 42)
    
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_classifier.fit(X_train, y_train)
    
    predictions = rf_classifier.predict(X_test)
    
    accuracy = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, average='macro')
    print("Test Accuracy/F1 for {}: {:.2f}% / {}".format(y_label, accuracy * 100,f1))
    
    #print("\nClassification Report:\n", classification_report(y_test, predictions))
    #print("\nConfusion Matrix:\n", confusion_matrix(y_test, predictions))

    return [rf_classifier, predictions]

In [13]:
def run_logistic(df, y_label):
    X = df.drop(columns=y_label)
    #trans = MinMaxScaler()
    trans = RobustScaler()
    X = trans.fit_transform(X)
    #X = DataFrame(X)
    
    y = df[y_label]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = 42)
    
    logreg_classifier = LogisticRegression(random_state=42)

    logreg_classifier.fit(X_train, y_train)

    predictions = logreg_classifier.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, average='macro')
    print("Test Accuracy/F1 for {}: {:.2f}% / {}".format(y_label, accuracy * 100,f1))
    #print("\nClassification Report:\n", classification_report(y_test, predictions))
    #print("\nConfusion Matrix:\n", confusion_matrix(y_test, predictions))
    return [logreg_classifier, predictions]

In [14]:
def run_seq_nn(df, y_label):
    X = df.drop(columns=y_label)
    #trans = MinMaxScaler()
    trans = RobustScaler()
    X = trans.fit_transform(X)
    #X = DataFrame(X)
    
    y = df[y_label]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = 42)
    model = Sequential()

    model.add(Dense(64, input_dim=X_train.shape[1], activation='relu'))
    model.add(Dense(32, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

    predictions = (model.predict(X_test) > 0.5).astype(int)

    accuracy = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, average='macro')
    print("Test Accuracy/F1 for {}: {:.2f}% / {}".format(y_label, accuracy * 100,f1))

    #print("\nClassification Report:\n", classification_report(y_test, predictions))
    #print("\nConfusion Matrix:\n", confusion_matrix(y_test, predictions))
    return [model, predictions]

In [15]:
def run_GBM(df, y_label):
    X = df.drop(columns=y_label)
    #trans = MinMaxScaler()
    trans = RobustScaler()
    X = trans.fit_transform(X)
    
    y = df[y_label]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = 42)
    
    gbm_classifier = GradientBoostingClassifier(n_estimators=100, random_state=42)
    
    gbm_classifier.fit(X_train, y_train)
    
    predictions = gbm_classifier.predict(X_test)
    
    accuracy = accuracy_score(y_test, predictions)
    
    accuracy = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, average='macro')
    print("Test Accuracy/F1 for {}: {:.2f}% / {}".format(y_label, accuracy * 100,f1))

    #print("\nClassification Report:\n", classification_report(y_test, predictions))
    #print("\nConfusion Matrix:\n", confusion_matrix(y_test, predictions))
    return [gbm_classifier, predictions]

In [16]:
def get_predictions(model_type):
    predictions = {}
    model = {}
    if model_type == 'naive':
        for label in labels:
            if label == 'comedy':
                result = run_naive_bayes(df_comedy, label)
                predictions['comedy'] = result[1]
                model['comedy'] = result[0]
            elif label == 'action':
                result = run_naive_bayes(df_action, label)
                predictions['action'] = result[1]
                model['action'] = result[0]
            elif label == 'drama':
                result = run_naive_bayes(df_drama, label)
                predictions['drama'] = result[1]
                model['drama'] = result[0]
            elif label == 'horror':
                result = run_naive_bayes(df_horror, label)
                predictions['horror'] = result[1]
                model['horror'] = result[0]
            elif label == 'romance':
                result = run_naive_bayes(df_romance, label)
                predictions['romance'] = result[1]
                model['romance'] = result[0]
            elif label == 'sci-fi':
                result = run_naive_bayes(df_sci_fi, label)
                predictions['sci-fi'] = result[1]
                model['sci-fi'] = result[0]
    elif model_type == 'logistic':
        for label in labels:
            if label == 'comedy':
                result = run_logistic(df_comedy, label)
                predictions['comedy'] = result[1]
                model['comedy'] = result[0]
            elif label == 'action':
                result = run_logistic(df_action, label)
                predictions['action'] = result[1]
                model['action'] = result[0]
            elif label == 'drama':
                result = run_logistic(df_drama, label)
                predictions['drama'] = result[1]
                model['drama'] = result[0]
            elif label == 'horror':
                result = run_logistic(df_horror, label)
                predictions['horror'] = result[1]
                model['horror'] = result[0]
            elif label == 'romance':
                result = run_logistic(df_romance, label)
                predictions['romance'] = result[1]
                model['romance'] = result[0]
            elif label == 'sci-fi':
                result = run_logistic(df_sci_fi, label)
                predictions['sci-fi'] = result[1]
                model['sci-fi'] = result[0]
    elif model_type == 'random_forrest':
        for label in labels:
            if label == 'comedy':
                result = run_rf(df_comedy, label)
                predictions['comedy'] = result[1]
                model['comedy'] = result[0]
            elif label == 'action':
                result = run_rf(df_action, label)
                predictions['action'] = result[1]
                model['action'] = result[0]
            elif label == 'drama':
                result = run_rf(df_drama, label)
                predictions['drama'] = result[1]
                model['drama'] = result[0]
            elif label == 'horror':
                result = run_rf(df_horror, label)
                predictions['horror'] = result[1]
                model['horror'] = result[0]
            elif label == 'romance':
                result = run_rf(df_romance, label)
                predictions['romance'] = result[1]
                model['romance'] = result[0]
            elif label == 'sci-fi':
                result = run_rf(df_sci_fi, label)
                predictions['sci-fi'] = result[1]
                model['sci-fi'] = result[0]
    elif model_type == 'seq_nn':
        for label in labels:
            if label == 'comedy':
                result = run_seq_nn(df_comedy, label)
                predictions['comedy'] = result[1]
                model['comedy'] = result[0]
            elif label == 'action':
                result = run_seq_nn(df_action, label)
                predictions['action'] = result[1]
                model['action'] = result[0]
            elif label == 'drama':
                result = run_seq_nn(df_drama, label)
                predictions['drama'] = result[1]
                model['drama'] = result[0]
            elif label == 'horror':
                result = run_seq_nn(df_horror, label)
                predictions['horror'] = result[1]
                model['horror'] = result[0]
            elif label == 'romance':
                result = run_seq_nn(df_romance, label)
                predictions['romance'] = result[1]
                model['romance'] = result[0]
            elif label == 'sci-fi':
                result = run_seq_nn(df_sci_fi, label)
                predictions['sci-fi'] = result[1]
                model['sci-fi'] = result[0]
    elif model_type == 'GBM':
        for label in labels:
            if label == 'comedy':
                result = run_GBM(df_comedy, label)
                predictions['comedy'] = result[1]
                model['comedy'] = result[0]
            elif label == 'action':
                result = run_GBM(df_action, label)
                predictions['action'] = result[1]
                model['action'] = result[0]
            elif label == 'drama':
                result = run_GBM(df_drama, label)
                predictions['drama'] = result[1]
                model['drama'] = result[0]
            elif label == 'horror':
                result = run_GBM(df_horror, label)
                predictions['horror'] = result[1]
                model['horror'] = result[0]
            elif label == 'romance':
                result = run_GBM(df_romance, label)
                predictions['romance'] = result[1]
                model['romance'] = result[0]
            elif label == 'sci-fi':
                result = run_GBM(df_sci_fi, label)
                predictions['sci-fi'] = result[1]
                model['sci-fi'] = result[0]
    return {'predictions': predictions, model_type+'_model':model}

In [17]:
naive_result = get_predictions('naive') #with duration_ms

Test Accuracy/F1 for action: 63.28% / 0.6139440286365115
Test Accuracy/F1 for comedy: 71.35% / 0.585996198568873
Test Accuracy/F1 for drama: 58.83% / 0.37040280210157617
Test Accuracy/F1 for horror: 89.15% / 0.4713235294117647
Test Accuracy/F1 for romance: 88.73% / 0.4701547531319086
Test Accuracy/F1 for sci-fi: 65.51% / 0.3957983193277311


In [19]:
rf_result = get_predictions('random_forrest')

Test Accuracy/F1 for action: 71.21% / 0.7120444374793222
Test Accuracy/F1 for comedy: 74.83% / 0.6907791408564823
Test Accuracy/F1 for drama: 69.12% / 0.6443766152749308
Test Accuracy/F1 for horror: 88.73% / 0.4935959306117125
Test Accuracy/F1 for romance: 89.15% / 0.5560885258358662
Test Accuracy/F1 for sci-fi: 69.40% / 0.5982832182039821


In [21]:
logistic_result = get_predictions('logistic')

Test Accuracy/F1 for action: 65.23% / 0.6519979865252071
Test Accuracy/F1 for comedy: 73.57% / 0.6777579634661837
Test Accuracy/F1 for drama: 63.70% / 0.5721028545629913
Test Accuracy/F1 for horror: 89.01% / 0.47093451066961
Test Accuracy/F1 for romance: 88.18% / 0.5014154368274635
Test Accuracy/F1 for sci-fi: 66.06% / 0.4959315021261924


In [22]:
GBM_result = get_predictions('GBM') 

Test Accuracy/F1 for action: 74.27% / 0.7415463114268338
Test Accuracy/F1 for comedy: 74.41% / 0.6879340277777777
Test Accuracy/F1 for drama: 68.29% / 0.6380808562496688
Test Accuracy/F1 for horror: 88.73% / 0.5522728845222445
Test Accuracy/F1 for romance: 87.34% / 0.5325331332833209
Test Accuracy/F1 for sci-fi: 68.29% / 0.6121898597626753


In [23]:
# simple ff neural net is good to start for binary classification
seq_nn_result = get_predictions('seq_nn') #with duration_ms

Epoch 1/10
53/53 [==============================] - 1s 3ms/step - loss: 0.6643 - accuracy: 0.6084 - val_loss: 0.6381 - val_accuracy: 0.6384
Epoch 2/10
53/53 [==============================] - 0s 1ms/step - loss: 0.6094 - accuracy: 0.6710 - val_loss: 0.6297 - val_accuracy: 0.6495
Epoch 3/10
53/53 [==============================] - 0s 1ms/step - loss: 0.5813 - accuracy: 0.7003 - val_loss: 0.6220 - val_accuracy: 0.6565
Epoch 4/10
53/53 [==============================] - 0s 1ms/step - loss: 0.5662 - accuracy: 0.7039 - val_loss: 0.6110 - val_accuracy: 0.6606
Epoch 5/10
53/53 [==============================] - 0s 1ms/step - loss: 0.5563 - accuracy: 0.7200 - val_loss: 0.6060 - val_accuracy: 0.6773
Epoch 6/10
53/53 [==============================] - 0s 1ms/step - loss: 0.5474 - accuracy: 0.7212 - val_loss: 0.6162 - val_accuracy: 0.6676
Epoch 7/10
53/53 [==============================] - 0s 1ms/step - loss: 0.5424 - accuracy: 0.7224 - val_loss: 0.6112 - val_accuracy: 0.6759
Epoch 8/10
53/53 [==